# 0. Setup & Configuration 

In [5]:
# Python imports
import sys
import numpy as np
import pandas as pd
import ast
from pathlib import Path
import joblib

# WFDB
import wfdb

# Sklearn imports
from sklearn.preprocessing import MultiLabelBinarizer

# Pytorch imports 
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Plotting 
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
sys.path.append(str(Path.cwd().parent / 'modules'))

# Paths
ROOT       = Path.cwd().parent
DATA       = ROOT / 'data'
ARTIFACTS  = ROOT / 'artifacts'

# Constants
CLASS_ORDER = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
FS = 100
SEED = 42

# Load pos_weight computed in EDA
pos_weight = np.load(ARTIFACTS / 'pos_weight.npy')
print(f"pos_weight: {dict(zip(CLASS_ORDER, pos_weight.round(2)))}")

pos_weight: {'NORM': np.float64(1.25), 'MI': np.float64(2.9), 'STTC': np.float64(3.08), 'CD': np.float64(3.37), 'HYP': np.float64(7.06)}


In [7]:
# Custom modules 
from preprocessing import preprocess

In [8]:
from utils import get_device, set_seed

SEED        = 42
CLASS_ORDER = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
FS          = 100
BATCH_SIZE  = 64

set_seed(SEED)
DEVICE = get_device()

Using device: cuda
GPU: NVIDIA GeForce RTX 3080
VRAM: 10.7 GB


# 1. Data Pipeline
## 1.1 Dataset Class (PTBXL Dataset)

In [9]:
### Load Metadata and rebuild splits
# Load dataset
df = pd.read_csv(DATA / 'ptbxl_database.csv', index_col='ecg_id')
df["scp_codes"] = df["scp_codes"].apply(ast.literal_eval) # Convert the scp codes into actual dictionaries

agg = pd.read_csv(DATA / 'scp_statements.csv', index_col=0) # Load the scp codes 
agg = agg[agg["diagnostic"] == 1]

# Define helper function to define the superclass
def get_superclass(scp_dict):
    return list(set(
        agg.loc[code, 'diagnostic_class']
        for code in scp_dict
        if code in agg.index
    ))
# Define the superclass based on scp code
df['superclass'] = df.scp_codes.apply(get_superclass)
df = df[df['superclass'].map(len) > 0]

# Splits as defined by Physionet
df['split'] = 'train'
df.loc[df['strat_fold'] == 9,  'split'] = 'val'
df.loc[df['strat_fold'] == 10, 'split'] = 'test'

# Binarize labels
mlb = MultiLabelBinarizer(classes=CLASS_ORDER)
Y = mlb.fit_transform(df['superclass'])
Y_df = pd.DataFrame(Y, index=df.index, columns=CLASS_ORDER)

print(f"Total records: {len(df):,}")
print(f"Label matrix shape: {Y.shape}")

Total records: 21,388
Label matrix shape: (21388, 5)


### Dataset Class

In [10]:
# Create Pytorch dataset class
class PTBXLDataset(Dataset):
    """
    PyTorch Dataset for PTB-XL 12-lead ECG classification.

    Loads raw waveforms, applies the full preprocessing
    pipeline (bandpass filter + per-record z-score normalization),
    and returns tensors.

    Args:
        df (pd.DataFrame): Dataframe subset for this split containing
                           'filename_lr' or 'filename_hr' columns.
        labels (np.ndarray): Multi-label binary matrix of shape (N, 5).
        ptbxl_root (Path): Path to PTB-XL root directory.
        fs (int): Sampling frequency — 100 or 500 Hz. Default 100.
        augment (bool): Whether to apply data augmentation. Default False.

    Returns per item:
        signal (torch.Tensor): Shape (12, 1000) float32 — (leads, timesteps).
        label (torch.Tensor): Shape (5,) float32 — multi-label binary vector.
    """

    def __init__(self, df: pd.DataFrame, labels: np.ndarray, ptbxl_root: Path, fs: int = 100, augment: bool = False):
        """Initialize arguments; stores everything the dataset needs to do its job"""
        self.df = df.reset_index() # ensures integer positional indexing works correctly when __getitem__ uses .iloc[idx]
        self.labels = labels # Labels for the Data
        self.ptbxl_root = ptbxl_root # Path() to data
        self.key = 'filename_lr' if fs == 100 else 'filename_hr' # Default to 100 Hz data if not use 500 Hz data
        self.fs = fs # Sampling frequency
        self.augment = augment # Augment the data True or False
    
    def __len__(self) -> int:
        """Tells PyTorch how many records are in the dataset. The DataLoader uses this to know when one epoch is complete"""
        return len(self.df)
    
    def __getitem__(self, idx: int) -> tuple:
        """
        Core method that lets PyTorch Train.
        For each idx 5 processes occur.
            1. Finds file path - looks up `filename_lr` in the dataframe.
            2. Loads the waveform - `wfdb.rdsamp` reads the `.dat` and `.hea` files and returns a numpy array.
            3. Pre-process - runs the bandpass filter then z-score normalization.
            4. Transposes - flips from (1000, 12) to (12, 1000) because PyTorch accepts layers in (channels, timesteps).
            5. Return Tensors - signal and label as `float32` tensors.
        """
        # Index the Signal row
        row = self.df.iloc[idx]

        # Load raw signal — shape: (1000, 12) when using 100 Hz
        signal, _ = wfdb.rdsamp(str(self.ptbxl_root / row[self.key]))
        signal    = np.array(signal, dtype=np.float32)

        # Preprocess: bandpass filter + per-record z-score
        signal = preprocess(signal, fs=self.fs)

        # Transpose to (12, 1000) for Conv1d — (leads, timesteps)
        signal = signal.T

        # Optional augmentation (training only)
        if self.augment:
            signal = self._augment(signal)

        label = self.labels[idx].astype(np.float32)

        return torch.tensor(signal), torch.tensor(label)
    
    def _augment(self, signal: np.ndarray) -> np.ndarray:
        """
        Helper function for light augmentation for training purposes. 
        Applied per-record at load time.
        """
        # Apply Gausian noise 
        if np.random.rand() < 0.5: 
            signal = signal + np.random.normal(0, 0.01, signal.shape).astype(np.float32)
        
        # Random Amplitude scaling 
        if np.random.rand() < 0.5: 
            scale = np.random.uniform(0.9, 1.1)
            signal = signal * scale
        
        return signal

In [11]:
# Subsets
train_df = df[df['split'] == 'train']
val_df   = df[df['split'] == 'val']
test_df  = df[df['split'] == 'test']

Y_train = Y[df['split'].values == 'train']
Y_val   = Y[df['split'].values == 'val']
Y_test  = Y[df['split'].values == 'test']

# Datasets
train_dataset = PTBXLDataset(train_df, Y_train, DATA, fs=FS, augment=True)
val_dataset   = PTBXLDataset(val_df,   Y_val,   DATA, fs=FS, augment=False)
test_dataset  = PTBXLDataset(test_df,  Y_test,  DATA, fs=FS, augment=False)

# DataLoaders; to test them first lets use batch 1
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

Train batches: 534
Val batches:   68
Test batches:  68


In [12]:
# Sanity check — inspect one batch
signals, labels = next(iter(train_loader))

print(f"Signal shape:  {signals.shape}")   # expect (32, 12, 1000)
print(f"Label shape:   {labels.shape}")    # expect (32, 5)
print(f"Signal dtype:  {signals.dtype}")   # expect torch.float32
print(f"Label dtype:   {labels.dtype}")    # expect torch.float32
print(f"Signal mean:   {signals.mean():.4f}")   # expect close to 0
print(f"Signal std:    {signals.std():.4f}")    # expect close to 1
print(f"Label sum:     {labels.sum(dim=1)}")     # labels per record
assert not torch.isnan(signals).any(), "NaN in signals"
assert not torch.isnan(labels).any(),  "NaN in labels"
print("Batch sanity check passed")

Signal shape:  torch.Size([32, 12, 1000])
Label shape:   torch.Size([32, 5])
Signal dtype:  torch.float32
Label dtype:   torch.float32
Signal mean:   0.0000
Signal std:    0.9952
Label sum:     tensor([1., 1., 1., 2., 1., 1., 1., 1., 1., 1., 1., 2., 1., 1., 1., 1., 2., 2.,
        1., 1., 1., 3., 1., 1., 2., 1., 1., 1., 1., 1., 1., 1.])
Batch sanity check passed


# Modeling Goal

We want to create models that can predict multilabel classification 
['CD', 'HYP, 'MI', 'NORM', 'STTC]

# Baseline Models

## Logistic and Ensemble Models
Logistic Regression is a very well known classifier. To be able to use such a classifier we can't pass in a raw waveform. In order to create this baseline model in addition to ensemble models, we need to feature engineer some features about the signals. 

We can feature engineer traits about them such as the following:
1. Mean 
2. Standard Deviation 
3. Minimum
4. Maximum
5. Range
6. Absolute Mean
7. RMS
8. Skewness
9. Kurtosis
10. Percentiles

With those traits build a feature matrix to train the models.

In [13]:
# modules/features.py

"""
Feature extraction for classical baseline models.
Extracts per-lead statistical features from preprocessed ECG signals.
"""

import numpy as np
import pandas as pd
import wfdb
from pathlib import Path
from scipy.stats import skew, kurtosis as kurt
from tqdm import tqdm

from preprocessing import preprocess


def extract_features(signal: np.ndarray) -> np.ndarray:
    """
    Extract per-lead statistical features from a preprocessed ECG signal.

    Computes 13 statistical features per lead across all 12 leads,
    producing a 156-dimensional feature vector per record.

    Features per lead:
        mean, std, min, max, range, absolute mean, RMS,
        skewness, kurtosis, p10, p25, p75, p90

    Args:
        signal (np.ndarray): Preprocessed ECG of shape (timesteps, 12).

    Returns:
        np.ndarray: Feature vector of shape (156,) float32.

    Example:
        >>> signal = preprocess(raw_signal)   # (1000, 12)
        >>> features = extract_features(signal)
        >>> features.shape
        (156,)
    """
    features = []
    for i in range(signal.shape[1]):
        lead = signal[:, i]
        features.extend([
            lead.mean(),
            lead.std(),
            lead.min(),
            lead.max(),
            lead.max() - lead.min(),      # range
            np.abs(lead).mean(),          # absolute mean
            np.sqrt(np.mean(lead ** 2)), # RMS
            skew(lead),                   # skewness
            kurt(lead),                   # kurtosis
            np.percentile(lead, 10),
            np.percentile(lead, 25),
            np.percentile(lead, 75),
            np.percentile(lead, 90),
        ])
    return np.array(features, dtype=np.float32)


def build_feature_matrix(df: pd.DataFrame, path: Path,
                          fs: int = 100) -> np.ndarray:
    """
    Build a feature matrix by extracting statistical features from
    all records in a dataframe split.

    Loads each record from disk, applies the full preprocessing pipeline,
    then extracts statistical features. Uses the same preprocessing as the
    deep learning model for fair comparison.

    Args:
        df (pd.DataFrame): Dataframe split with filename columns.
        path (Path): Path to PTB-XL root directory.
        fs (int): Sampling frequency. Default 100.

    Returns:
        np.ndarray: Feature matrix of shape (N, 156).

    Example:
        >>> X_train = build_feature_matrix(train_df, PTBXL_ROOT, fs=100)
        >>> X_train.shape
        (17440, 156)
    """
    key      = 'filename_lr' if fs == 100 else 'filename_hr'
    features = []

    for _, row in tqdm(df.iterrows(), total=len(df),
                       desc='Extracting features'):
        signal, _ = wfdb.rdsamp(str(path / row[key]))
        signal    = np.array(signal, dtype=np.float32)
        signal    = preprocess(signal, fs=fs)
        features.append(extract_features(signal))

    return np.stack(features)  # shape: (N, 156)

In [14]:
print("Extracting features — this will take a few minutes...")

# X_train_feat = build_feature_matrix(train_df, DATA, fs=FS)
# X_val_feat   = build_feature_matrix(val_df,   DATA, fs=FS)
# X_test_feat  = build_feature_matrix(test_df,  DATA, fs=FS)
X_train_feat = np.load(ARTIFACTS / 'X_train_feat.npy')
X_val_feat = np.load(ARTIFACTS / 'X_val_feat.npy')
X_test_feat = np.load(ARTIFACTS / 'X_test_feat.npy')

print(f"Train: {X_train_feat.shape}")  # expect (17440, 156)
print(f"Val:   {X_val_feat.shape}")    # expect (2179, 156)
print(f"Test:  {X_test_feat.shape}")   # expect (2179, 156)

# # Save so we don't need to recompute them
# np.save(ARTIFACTS / 'X_train_feat.npy', X_train_feat)
# np.save(ARTIFACTS / 'X_val_feat.npy',   X_val_feat)
# np.save(ARTIFACTS / 'X_test_feat.npy',  X_test_feat)
# print("Features saved to artifacts/")

Extracting features — this will take a few minutes...
Train: (17084, 156)
Val:   (2146, 156)
Test:  (2158, 156)


### Approach for Model Training 
1. Scale features using `StandardScaler` for train only. 
2. Train `LogisticRegression` with `OneVsRestClassifier` for multi-label.
3. Train `RandomForestClassifier` with `MultiOutputClassifier`.
4. Evaluate both on the test set with macro-AUC.
5. Save predicted probablities to `artifacts` for the DeLong test later.

In [15]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import roc_auc_score

# Fit on Training data and transform on test and val
scaler =  StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_feat)
X_test_scaled = scaler.transform(X_test_feat)
X_val_scaled = scaler.transform(X_val_feat)

# Train the Logistic Classifier 
print("Training Logistic Classifier ...")
log = OneVsRestClassifier(
    LogisticRegression(
        random_state=SEED,
    n_jobs = -1 # Use all CPU cores
    )
)
log.fit(X_train_scaled, Y_train)

# Save model
joblib.dump(log, ARTIFACTS / 'logistic_regression.pkl')
print("Logistic Regression saved to artifacts/")

# Quick validation AUC
log_val_probs = log.predict_proba(X_val_scaled)
log_val_auc   = roc_auc_score(Y_val, log_val_probs, average='macro')
print(f"Logistic Regression — Val Macro-AUC: {log_val_auc:.4f}")

Training Logistic Classifier ...
Logistic Regression saved to artifacts/
Logistic Regression — Val Macro-AUC: 0.8654


In [16]:
# np.save(ARTIFACTS / 'log_val_probs.npy',  log_val_probs)
# np.save(ARTIFACTS / 'log_test_probs.npy', log.predict_proba(X_test_scaled))
# print("LR probabilities saved to artifacts/")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier

print("Training Random Forest...")
rf = MultiOutputClassifier(
    RandomForestClassifier(
        n_estimators = 200,
        max_depth    = None,   # grow full trees
        min_samples_split = 5,
        n_jobs       = -1,     # use all CPU cores
        random_state = SEED
    ),
    n_jobs = -1
)
rf.fit(X_train_scaled, Y_train)

# # Save model
# joblib.dump(rf, ARTIFACTS / 'random_forest.pkl')
# print("Random Forest saved to artifacts/")

# Validation AUC
rf_val_probs = np.stack([
    est.predict_proba(X_val_scaled)[:, 1]
    for est in rf.estimators_
], axis=1)

rf_val_auc = roc_auc_score(Y_val, rf_val_probs, average='macro')
print(f"Random Forest — Val Macro-AUC: {rf_val_auc:.4f}")

Training Random Forest...
Random Forest saved to artifacts/
Random Forest — Val Macro-AUC: 0.8753


In [18]:
# np.save(ARTIFACTS / 'rf_val_probs.npy',  rf_val_probs)
# np.save(ARTIFACTS / 'rf_test_probs.npy', np.stack([
#     est.predict_proba(X_test_scaled)[:, 1]
#     for est in rf.estimators_
# ], axis=1))
# print("RF probabilities saved to artifacts/")

## Deep Learning Model
### ResNET 1D Architecture

An ECG is a time series — voltage measured over time across 12 leads. It has one spatial dimension (time) not two like an image. ResNet2D (standard ResNet used for images) uses 2D convolutions that slide across height and width. That makes no sense for a signal where the meaningful patterns are temporal — QRS complexes, ST segments, P waves — all of which are features in the time dimension only.

**Why Convolutional?**
Convolutional networks are well suited to ECG because the diagnostic patterns you care about are local and translation invariant. A QRS complex looks the same whether it occurs at second 2 or second 7 of the recording. A conv filter that learns to detect a QRS complex can find it anywhere in the signal — this is exactly what convolutions are designed to do.

The Strodthoff et al. (2021) PTB-XL benchmark paper specifically evaluates ResNet1D and reports it as one of the strongest performing architectures on this dataset. Using it can strengthen our validation report.

In [22]:
# Create RESNet inspired architecture 
import torch
import torch.nn as nn

class ResBlock1D(nn.Module):
    """
    Single 1D block with two convolutional layers and a skip connection.

    Args:
        in_channels (int): Number of input channels.
        out_channels (int): Number of output channels.
        kernel_size (int): Convolutional kernel size. Default 7.
        stride (int): Stride for first convolution. Default 1.
        dropout (float): Dropout probability. Default 0.2.
    """
    def __init__ (self, in_channels: int, out_channels: int, kernel_size: int = 7, stride: int = 1, dropout: float = 0.2):
        super().__init__() # Adopt properties of nn.Module

        padding = kernel_size // 2 # Should be 1

        # Define the blocks
        self.main = nn.Sequential(
            nn.Conv1d(
                in_channels=in_channels, out_channels=out_channels, 
                kernel_size=kernel_size, stride=stride, padding=padding
            ),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Conv1d(
                in_channels=out_channels, out_channels=out_channels, 
                kernel_size=kernel_size, stride=1, padding=padding
            ),
            nn.BatchNorm1d(out_channels),
        )
        # Skip path — Sequential if projection needed, Identity otherwise
        self.skip = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=1,
                      stride=stride, bias=False),
            nn.BatchNorm1d(out_channels)
        ) if (in_channels != out_channels or stride != 1) else nn.Identity()

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # This addition is why forward() is unavoidable in ResBlock
        return self.relu(self.main(x) + self.skip(x))

# Now we will define the entire ResNet Architecture
class ResNet1D(nn.Module):
    def __init__(self, n_leads: int = 12, n_classes: int = 5,
                 base_filters: int = 64, dropout: float = 0.2):
        super().__init__()
        f = base_filters

        # Stem — fully Sequential
        self.stem = nn.Sequential(
            nn.Conv1d(n_leads, f, kernel_size=15,
                      stride=2, padding=7, bias=False),
            nn.BatchNorm1d(f),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        )

        # 8 blocks — Sequential of ResBlock1D objects
        self.blocks = nn.Sequential(
            ResBlock1D(f, f, dropout=dropout),  # block 1
            ResBlock1D(f, f, dropout=dropout),  # block 2
            ResBlock1D(f, f * 2, stride=2, dropout=dropout),  # block 3
            ResBlock1D(f * 2, f * 2, dropout=dropout),  # block 4
            ResBlock1D(f * 2, f * 4, stride=2, dropout=dropout),  # block 5
            ResBlock1D(f * 4, f * 4, dropout=dropout),  # block 6
            ResBlock1D(f * 4, f * 8, stride=2, dropout=dropout),  # block 7
            ResBlock1D(f * 8, f * 8, dropout=dropout),  # block 8
        )

        # Head — fully Sequential
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(f * 8, n_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.blocks(x)
        x = self.head(x)
        return x

In [24]:
def count_parameters(model: nn.Module) -> int:
    """Count trainable parameters in a model."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
import torch.optim as optim

# -- Constants -----------------------------------------------------------------
N_LEADS = 12 
N_CLASSES = 5

# -- Loss function -------------------------------------------------------------
pos_weight_tensor = torch.tensor(pos_weight).to(DEVICE)
criterion         = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

# -- Define model --------------------------------------------------------------
model = ResNet1D(
    n_leads = N_LEADS,
    n_classes = N_CLASSES,
).to(DEVICE)

print(f"Parameters: {count_parameters(model):,}")
# -- Optimizer -----------------------------------------------------------------
optimizer = optim.Adam(
    model.parameters(),
    lr = 1e-3,
    weight_decay = 1e-4
)

# -- Learning rate scheduler ---------------------------------------------------
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode = 'min',     # reduce when val loss stops decreasing
    factor = 0.5,       # halve the lr
    patience = 5,         # wait 5 epochs before reducing
    verbose = True
)

# -- Early stopping ------------------------------------------------------------
class EarlyStopping:
    """
    Stop training when validation loss stops improving.

    Args:
        patience (int): Epochs to wait before stopping. Default 10.
        delta (float): Minimum improvement to count as improvement. Default 1e-4.
        path (Path): Where to save the best model checkpoint.
    """
    def __init__(self, patience: int = 10, delta: float = 1e-4,
                 path: Path = ARTIFACTS / 'best_model.pt'):
        self.patience   = patience
        self.delta      = delta
        self.path       = path
        self.counter    = 0
        self.best_loss  = np.inf
        self.early_stop = False

    def __call__(self, val_loss: float, model: nn.Module) -> None:
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter   = 0
            torch.save(model.state_dict(), self.path)
            print(f"  Checkpoint saved (val_loss={val_loss:.4f})")
        else:
            self.counter += 1
            print(f"  No improvement ({self.counter}/{self.patience})")
            if self.counter >= self.patience:
                self.early_stop = True

early_stopping = EarlyStopping(patience=10, path=ARTIFACTS / 'best_model.pt')

# -- History ---------------------------------------------------------------
history = {
    'train_loss': [],
    'val_loss':   [],
    'val_auc':    []
}

# ── Training loop ─────────────────────────────────────────────────────────────
def train_epoch(model, loader, criterion, optimizer, device):
    """Run one training epoch. Returns mean loss."""
    model.train()
    total_loss = 0.0

    for signals, labels in loader:
        signals = signals.to(device)
        labels  = labels.to(device)

        optimizer.zero_grad()
        logits = model(signals)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * signals.size(0)

    return total_loss / len(loader.dataset)


def val_epoch(model, loader, criterion, device):
    """Run one validation epoch. Returns mean loss, all probs, all labels."""
    model.eval()
    total_loss = 0.0
    all_probs  = []
    all_labels = []

    with torch.no_grad():
        for signals, labels in loader:
            signals = signals.to(device)
            labels  = labels.to(device)

            logits = model(signals)
            loss   = criterion(logits, labels)
            probs  = torch.sigmoid(logits)

            total_loss += loss.item() * signals.size(0)
            all_probs.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_probs  = np.concatenate(all_probs,  axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    mean_loss  = total_loss / len(loader.dataset)
    macro_auc  = roc_auc_score(all_labels, all_probs, average='macro')

    return mean_loss, macro_auc, all_probs, all_labels


# ── Run training ──────────────────────────────────────────────────────────────
EPOCHS = 50

print(f"Training ResNet1D for up to {EPOCHS} epochs")
print(f"Device: {DEVICE}")
print("=" * 60)

for epoch in range(1, EPOCHS + 1):
    train_loss               = train_epoch(model, train_loader, criterion,
                                           optimizer, DEVICE)
    val_loss, val_auc, _, _  = val_epoch(model, val_loader, criterion, DEVICE)

    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_auc)

    print(f"Epoch {epoch:02d}/{EPOCHS} | "
          f"train loss: {train_loss:.4f} | "
          f"val loss: {val_loss:.4f} | "
          f"val AUC: {val_auc:.4f}")

    early_stopping(val_loss, model)
    if early_stopping.early_stop:
        print(f"\nEarly stopping triggered at epoch {epoch}")
        break

print("\nTraining complete")

# ── Load best model ───────────────────────────────────────────────────────────
model.load_state_dict(torch.load(ARTIFACTS / 'best_model.pt',
                                  map_location=DEVICE))
print("Best model loaded from checkpoint")

# ── Plot training curves ──────────────────────────────────────────────────────
def plot_training_curves(history: dict) -> plt.Figure:
    """
    Plot training and validation loss curves plus validation AUC.

    Args:
        history (dict): Keys 'train_loss', 'val_loss', 'val_auc'.

    Returns:
        plt.Figure
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    epochs          = range(1, len(history['train_loss']) + 1)

    ax1.plot(epochs, history['train_loss'], color='#378ADD',
             linewidth=1.5, label='Train loss')
    ax1.plot(epochs, history['val_loss'],   color='#D4537E',
             linewidth=1.5, label='Val loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training and validation loss',
                  fontsize=12, fontweight='bold')
    ax1.legend()
    sns.despine(ax=ax1)

    ax2.plot(epochs, history['val_auc'], color='#1D9E75',
             linewidth=1.5, label='Val macro-AUC')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Macro-AUC')
    ax2.set_title('Validation macro-AUC',
                  fontsize=12, fontweight='bold')
    ax2.set_ylim(0.5, 1.0)
    ax2.legend()
    sns.despine(ax=ax2)

    fig.tight_layout()
    return fig

fig = plot_training_curves(history)
plt.show()

Parameters: 8,743,813
